In [ ]:
import os
import time
import numpy as np
import polars as pl
import optuna
import psutil
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow import keras
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

HAS_GPU = len(tf.config.list_physical_devices('GPU')) > 0
TRAIN_DEVICE = '/GPU:0' if HAS_GPU else '/CPU:0'
INFER_DEVICE = '/CPU:0'

if HAS_GPU:
    print('GPU detectada. El entrenamiento se ejecutara en GPU y la inferencia se medira en CPU.')
else:
    print('No hay GPU disponible. Entrenamiento e inferencia se ejecutaran en CPU.')

tf.keras.backend.clear_session()


In [ ]:
# ==========================================
# 1. FUNCIONES AUXILIARES Y CARGA DE DATOS
# ==========================================

def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps + 1):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps - 1])
    return np.array(Xs), np.array(ys)


DEFAULT_DROPOUT_RATE = 0.2


def build_cnn1d_model(time_steps, n_features, n_filters, kernel_size, dense_units, dropout_rate=DEFAULT_DROPOUT_RATE):
    model = keras.Sequential([
        keras.layers.Input(shape=(time_steps, n_features)),
        keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size, padding='same', activation='relu'),
        keras.layers.MaxPooling1D(pool_size=2),
        keras.layers.Conv1D(filters=max(16, n_filters // 2), kernel_size=kernel_size, padding='same', activation='relu'),
        keras.layers.GlobalMaxPooling1D(),
        keras.layers.Dense(dense_units, activation='relu'),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


def clone_model_to_cpu(trained_model, time_steps, n_features, n_filters, kernel_size, dense_units, dropout_rate):
    with tf.device(INFER_DEVICE):
        cpu_model = build_cnn1d_model(
            time_steps=time_steps,
            n_features=n_features,
            n_filters=n_filters,
            kernel_size=kernel_size,
            dense_units=dense_units,
            dropout_rate=dropout_rate
        )
        cpu_model.set_weights(trained_model.get_weights())
    return cpu_model

# ==========================================
# 2. PREPARACION DATASET
# ==========================================

TARGET_COL = "label"

df_encoded = pl.read_csv("../../DATASETS/dataSets_Reducidos/iot-23/datos_IOT_23_preparado.csv")

# Separación de características (X) y variable objetivo (y)
feature_columns = [col for col in df_encoded.columns if col != TARGET_COL]
X = df_encoded.select(feature_columns)
y_np = df_encoded[TARGET_COL].to_numpy().astype(np.int8)
X_np = X.to_numpy()

display(X.head())

indices = np.arange(X_np.shape[0])

train_full_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y_np,
)

train_idx, val_idx = train_test_split(
    train_full_idx,
    test_size=0.2,
    random_state=42,
    stratify=y_np[train_full_idx],
)

X_full_train_np = X_np[train_full_idx]
X_test_np = X_np[test_idx]
y_full_train = y_np[train_full_idx]
y_test_np = y_np[test_idx]

X_train_np = X_np[train_idx]
X_val_np = X_np[val_idx]
y_train_np = y_np[train_idx]
y_val_np = y_np[val_idx]

print(f"Entrenamiento: {len(X_train_np):,} muestras")
print(f"Validación:    {len(X_val_np):,} muestras")
print(f"Test:          {len(X_test_np):,} muestras")
print(f"Clases en test: {np.unique(y_test_np)}")
print(f"Total muestras: {len(X_np):,}")


In [ ]:
# ==========================================
# 2. OPTUNA MULTIOBJETIVO: F1 Y LATENCIA
# ==========================================

def objective(trial):
    tf.keras.backend.clear_session()

    time_steps = trial.suggest_int('time_steps', 5, 15)
    n_filters = trial.suggest_int('n_filters', 32, 128, step=32)
    kernel_size = trial.suggest_int('kernel_size', 2, min(5, time_steps))
    dense_units = trial.suggest_int('dense_units', 16, 96, step=16)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    f1_scores = []
    inference_times = []

    for train_idx, val_idx in skf.split(X_full_train_np, y_full_train):
        X_train_fold = X_full_train_np[train_idx]
        y_train_fold = y_full_train[train_idx]
        X_val_fold = X_full_train_np[val_idx]
        y_val_fold = y_full_train[val_idx]

        scaler = StandardScaler()
        X_train_fold_scaled = scaler.fit_transform(X_train_fold)
        X_val_fold_scaled = scaler.transform(X_val_fold)

        X_train_seq, y_train_seq = create_sequences(X_train_fold_scaled, y_train_fold, time_steps)
        X_val_seq, y_val_seq = create_sequences(X_val_fold_scaled, y_val_fold, time_steps)

        with tf.device(TRAIN_DEVICE):
            model = build_cnn1d_model(
                time_steps=time_steps,
                n_features=X_train_seq.shape[2],
                n_filters=n_filters,
                kernel_size=kernel_size,
                dense_units=dense_units,
                dropout_rate=DEFAULT_DROPOUT_RATE
            )

            early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
            model.fit(
                X_train_seq,
                y_train_seq,
                validation_split=0.1,
                epochs=20,
                batch_size=1024,
                callbacks=[early_stop],
                verbose=0
            )

        cpu_model = clone_model_to_cpu(
            trained_model=model,
            time_steps=time_steps,
            n_features=X_train_seq.shape[2],
            n_filters=n_filters,
            kernel_size=kernel_size,
            dense_units=dense_units,
            dropout_rate=DEFAULT_DROPOUT_RATE
        )

        with tf.device(INFER_DEVICE):
            y_pred_prob = cpu_model.predict(X_val_seq, batch_size=1024, verbose=0).ravel()

        y_pred = (y_pred_prob > 0.5).astype(np.int8)
        f1_scores.append(f1_score(y_val_seq, y_pred, zero_division=0))

        X_lat = X_val_seq[:min(20000, len(X_val_seq))]
        if len(X_lat) == 0:
            continue

        with tf.device(INFER_DEVICE):
            _ = cpu_model.predict(X_lat[:min(512, len(X_lat))], batch_size=512, verbose=0)

        rep_lat = []
        for _ in range(3):
            with tf.device(INFER_DEVICE):
                t0 = time.perf_counter()
                _ = cpu_model.predict(X_lat, batch_size=1024, verbose=0)
                t1 = time.perf_counter()
            rep_lat.append((t1 - t0) / len(X_lat))

        inference_times.append(float(np.mean(rep_lat)))

        tf.keras.backend.clear_session()

    return float(np.mean(f1_scores)), float(np.mean(inference_times))


study = optuna.create_study(directions=['maximize', 'minimize'], study_name='cnn1d_ids_optimization')
print('Iniciando barrido multiobjetivo con CNN-1D (entrenamiento en GPU si existe, inferencia medida en CPU)...')
study.optimize(objective, n_trials=25, show_progress_bar=True)

results = []
pareto_trials = {trial.number for trial in study.best_trials}
for trial in study.trials:
    if trial.values is None:
        continue

    row = {
        'Trial': trial.number,
        'F1_CV': float(trial.values[0]),
        'Latencia_ms': float(trial.values[1] * 1000),
        'Pareto': trial.number in pareto_trials
    }
    row.update(trial.params)
    results.append(row)

df_cnn_trials = pl.DataFrame(results).sort(['Pareto', 'F1_CV', 'Latencia_ms'], descending=[True, True, False])
df_cnn_trials.write_csv('cnn1d_trials_results_cv.csv')
print("\nResultados guardados en 'cnn1d_trials_results_cv.csv'")
print(df_cnn_trials)


In [ ]:
import polars as pl
import matplotlib.pyplot as plt

# Leer resultados desde el CSV
df_cnn_trials = pl.read_csv("cnn1d_trials_results_cv.csv")

# Filtrar frontera de Pareto
df_pareto = (
    df_cnn_trials
    .filter(pl.col("Pareto") == True)
    .sort("Latencia_ms")
)

display(df_pareto)

plt.figure(figsize=(10, 7))

plt.scatter(
    df_cnn_trials["Latencia_ms"],
    df_cnn_trials["F1_CV"],
    c="lightgray",
    label="Trials"
)

plt.scatter(
    df_pareto["Latencia_ms"],
    df_pareto["F1_CV"],
    c="red",
    label="Frontera de Pareto"
)

# Mostrar hiperparámetros de los puntos Pareto
for row in df_pareto.iter_rows(named=True):
    texto = (
        f"ts={row['time_steps']}, nf={row['n_filters']}\n"
        f"k={row['kernel_size']}, dense={row['dense_units']}"
    )
    plt.annotate(
        texto,
        (row["Latencia_ms"], row["F1_CV"]),
        textcoords="offset points",
        xytext=(6, 6),
        ha="left",
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.75)
    )

plt.xlabel("Latencia de inferencia (ms por muestra)")
plt.ylabel("F1 medio en CV")
plt.title("CNN-1D: F1 vs Latencia")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# ==========================================
# 4. EVALUACION FINAL EN TEST (CNN-1D)
# ==========================================

candidatos = [
    {'ts': 15, 'nf': 96, 'k': 3, 'd': 32, 'nombre': 'Candidato 1'},
    {'ts': 12, 'nf': 96, 'k': 3, 'd': 80, 'nombre': 'Candidato 2'},
    {'ts': 15, 'nf': 32, 'k': 4, 'd': 96, 'nombre': 'Candidato 3'},
    {'ts': 7, 'nf': 64, 'k': 4, 'd': 80, 'nombre': 'Candidato 4'},
    {'ts': 9, 'nf': 64, 'k': 3, 'd': 64, 'nombre': 'Candidato 5'},
]

resultados_finales = []

print('--- EVALUACION FINAL SOBRE EL SET DE TEST (CNN-1D) ---\n')

for c in candidatos:
    tf.keras.backend.clear_session()
    print(
        f"Probando {c['nombre']}: TS={c['ts']}, Filtros={c['nf']}, "
        f"Kernel={c['k']}, Dense={c['d']}"
    )

    time_steps = int(c['ts'])
    n_filters = int(c['nf'])
    kernel_size = int(c['k'])
    dense_units = int(c['d'])

    scaler = StandardScaler()
    X_full_train_scaled = scaler.fit_transform(X_full_train_np)
    X_test_scaled = scaler.transform(X_test_np)

    X_train_seq, y_train_seq = create_sequences(X_full_train_scaled, y_full_train, time_steps)
    X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_np, time_steps)

    with tf.device(TRAIN_DEVICE):
        model = build_cnn1d_model(
            time_steps=time_steps,
            n_features=X_train_seq.shape[2],
            n_filters=n_filters,
            kernel_size=kernel_size,
            dense_units=dense_units,
            dropout_rate=DEFAULT_DROPOUT_RATE
        )

        early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
        model.fit(
            X_train_seq,
            y_train_seq,
            validation_split=0.1,
            epochs=20,
            batch_size=1024,
            callbacks=[early_stop],
            verbose=0
        )

    cpu_model = clone_model_to_cpu(
        trained_model=model,
        time_steps=time_steps,
        n_features=X_train_seq.shape[2],
        n_filters=n_filters,
        kernel_size=kernel_size,
        dense_units=dense_units,
        dropout_rate=DEFAULT_DROPOUT_RATE
    )

    with tf.device(INFER_DEVICE):
        y_prob = cpu_model.predict(X_test_seq, batch_size=1024, verbose=0).ravel()
    y_pred = (y_prob > 0.5).astype(np.int8)

    X_lat = X_test_seq[:min(20000, len(X_test_seq))]
    if len(X_lat) == 0:
        raise ValueError('No hay suficientes muestras en test para medir latencia con la configuracion seleccionada.')

    with tf.device(INFER_DEVICE):
        _ = cpu_model.predict(X_lat[:min(512, len(X_lat))], batch_size=512, verbose=0)

    rep_lat = []
    for _ in range(3):
        with tf.device(INFER_DEVICE):
            t0 = time.perf_counter()
            _ = cpu_model.predict(X_lat, batch_size=1024, verbose=0)
            t1 = time.perf_counter()
        rep_lat.append((t1 - t0) / len(X_lat))

    f1_test = float(f1_score(y_test_seq, y_pred, zero_division=0))
    acc_test = float(accuracy_score(y_test_seq, y_pred))
    lat_ms = float(np.mean(rep_lat) * 1000)

    resultados_finales.append({
        'Perfil': c['nombre'],
        'time_steps': time_steps,
        'n_filters': n_filters,
        'kernel_size': kernel_size,
        'dense_units': dense_units,
        'F1_Test': f1_test,
        'Accuracy_Test': acc_test,
        'Latencia_ms': lat_ms
    })
    print(f"  -> F1={f1_test:.4f} | Acc={acc_test:.4f} | Latencia={lat_ms:.6f} ms")

df_candidate_results = pl.DataFrame(resultados_finales).sort(['F1_Test', 'Latencia_ms'], descending=[True, False])

print('\n' + '=' * 88)
print('              TABLA COMPARATIVA FINAL (CNN-1D - TEST SET)')
print('=' * 88)
print(df_candidate_results)

# Puedes cambiar manualmente esta eleccion si prefieres otro candidato.
best_c = candidatos[0]
print('\nModelo ganador seleccionado para ROC, matriz de confusion y benchmark:')
print(best_c)